# Notebook 03 — QLoRA Fine-tuning of the Generator (Qwen2.5-7B-Instruct)

**Pipeline position:** ablation row **C4** (the LLM fine-tuning step). DOCUMENTED NEGATIVE RESULT — production uses the base generator.

Trains on the shared `llm.jsonl` RAG-chat data (single-passage format):
1. Inspect the data; load Qwen2.5-7B in 4-bit NF4; attach LoRA (r=32, alpha=64).
2. Render with the Qwen chat template, filter >1024 tokens, 95/5 split.
3. Dry-run (50 steps) then the real run (1 epoch, cosine, paged_adamw_8bit) -> adapter `qwen-qlora-shared-v1/adapter_best`.

**Why it regressed (-0.065 Token F1):** the training data is single-passage but inference is multi-passage [Kaynak 1..N], so the model anchored to the wrong passage (faithfulness rose +24pp, Token F1 fell). The fix is multi-passage RAG-format training. (Recovered from the original Colab session; this is the actual C4 adapter.)


In [ ]:
# Inspect shared llm.jsonl, check schema, count tokens, estimate steps.
import json, torch
from pathlib import Path
from collections import Counter

LLM_PATH = Path("/content/drive/MyDrive/hukuk-rag/data/external/shared_2026/llm.jsonl")
recs = [json.loads(l) for l in open(LLM_PATH) if l.strip()]
print(f"Total training records: {len(recs)}")
print(f"\nSchema sample (first record):")
print(json.dumps(recs[0], ensure_ascii=False, indent=2)[:1500])
print("\nFirst record content lengths:")
for m in recs[0]["messages"]:
    print(f"  {m['role']:<12} {len(m['content'])} chars")

# Role distribution
role_counts = Counter()
for r in recs:
    for m in r["messages"]:
        role_counts[m["role"]] += 1
print(f"\nRole distribution across all messages: {dict(role_counts)}")

# Length stats
import numpy as np
char_lens = [sum(len(m["content"]) for m in r["messages"]) for r in recs]
print(f"\nTotal-content-chars per record: median {int(np.median(char_lens))}, p95 {int(np.percentile(char_lens,95))}, p99 {int(np.percentile(char_lens,99))}, max {max(char_lens)}")

free, total = torch.cuda.mem_get_info()
print(f"\nGPU free / total: {free/1e9:.1f} / {total/1e9:.1f} GB  (clean restart ✓)")


In [ ]:
# Load Qwen2.5-7B-Instruct in 4-bit, prepare for k-bit training, attach LoRA.
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE = "Qwen/Qwen2.5-7B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer + base model in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",   # safer with grad checkpointing
)
model.config.use_cache = False     # required for training
model.config.pretraining_tp = 1
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_cfg = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

free, total = torch.cuda.mem_get_info()
print(f"\nGPU free / total: {free/1e9:.1f} / {total/1e9:.1f} GB after model load")


In [ ]:
# Prep dataset: apply Qwen chat template, build HF Dataset, split 95/5.
import json
from datasets import Dataset

LLM_PATH = "/content/drive/MyDrive/hukuk-rag/data/external/shared_2026/llm.jsonl"
MAX_SEQ_LEN = 1024

recs = [json.loads(l) for l in open(LLM_PATH) if l.strip()]

# Render each record to plain text via Qwen chat template
def to_text(rec):
    return {"text": tokenizer.apply_chat_template(rec["messages"], tokenize=False, add_generation_prompt=False)}

# Build raw dataset
raw_texts = [to_text(r) for r in recs]

# Filter very long records (>MAX_SEQ_LEN tokens) — they'd be right-truncated, cutting the assistant turn
keep, drop = [], 0
for t in raw_texts:
    if len(tokenizer(t["text"], truncation=False)["input_ids"]) <= MAX_SEQ_LEN:
        keep.append(t)
    else:
        drop += 1
print(f"Kept {len(keep)} / dropped {drop} ({100*drop/len(raw_texts):.1f}%) records longer than {MAX_SEQ_LEN} tokens")

ds = Dataset.from_list(keep).shuffle(seed=42)
split = ds.train_test_split(test_size=0.05, seed=42)
train_ds, val_ds = split["train"], split["test"]
print(f"train: {len(train_ds)}  val: {len(val_ds)}")
print(f"\nSample formatted text (first 500 chars):\n{train_ds[0]['text'][:500]}")


In [ ]:
# Setup SFTTrainer. DRY RUN: max_steps=50, no save, just verify training loop works + loss decreases.
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

OUT_DIR = "/content/drive/MyDrive/hukuk-rag/models/qwen-qlora-shared-v1"
DRY_RUN_DIR = "/content/qlora_dryrun"

dry_args = SFTConfig(
    output_dir=DRY_RUN_DIR,
    max_steps=50,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="constant",          # constant for dry run
    warmup_steps=5,
    weight_decay=0.01,
    optim="paged_adamw_8bit",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=5,
    save_strategy="no",
    eval_strategy="no",
    report_to="none",
    max_length=1024,
    packing=False,
    dataset_text_field="text",
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=dry_args,
    processing_class=tokenizer,
)

print("=== DRY RUN: 50 optimizer steps ===")
print(f"  effective batch: {dry_args.per_device_train_batch_size * dry_args.gradient_accumulation_steps}")
print(f"  steps × batch  : {dry_args.max_steps * dry_args.per_device_train_batch_size * dry_args.gradient_accumulation_steps} examples")
trainer.train()
print("\nDry-run complete.")


In [ ]:
# Free dry-run trainer state, then setup the real training run.
import gc, torch
del trainer
gc.collect()
torch.cuda.empty_cache()

from trl import SFTTrainer, SFTConfig

OUT_DIR = "/content/drive/MyDrive/hukuk-rag/models/qwen-qlora-shared-v1"

real_args = SFTConfig(
    output_dir=OUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    optim="paged_adamw_8bit",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    save_only_model=False,           # keep optimizer for resume
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    max_length=1024,
    packing=False,
    dataset_text_field="text",
    remove_unused_columns=False,
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=real_args,
    processing_class=tokenizer,
)

# Calculate total time estimate
total_optim_steps = len(train_ds) // (real_args.per_device_train_batch_size * real_args.gradient_accumulation_steps) * real_args.num_train_epochs
sec_per_step = 38  # measured in dry-run
print(f"=== FULL TRAINING START ===")
print(f"  effective batch    : {real_args.per_device_train_batch_size * real_args.gradient_accumulation_steps}")
print(f"  total optim steps  : {total_optim_steps}")
print(f"  est duration       : {total_optim_steps * sec_per_step / 3600:.1f} hours")
print(f"  output dir         : {OUT_DIR}")
print(f"  save every         : {real_args.save_steps} steps (~{real_args.save_steps*sec_per_step/60:.0f} min)")
print(f"  eval every         : {real_args.eval_steps} steps")
print()
trainer.train()

# Save final adapter explicitly (load_best_model_at_end will swap to best ckpt first)
print("\n=== TRAINING DONE — saving best adapter ===")
trainer.save_model(OUT_DIR + "/adapter_best")
tokenizer.save_pretrained(OUT_DIR + "/adapter_best")
print(f"Saved to: {OUT_DIR}/adapter_best")
